# 03. Otimização Bayesiana com Optuna (Frente 3.1 MLP)
## Trabalho Prático de Deep Learning (2026/2) — Prof. Dr. Sérgio Canuto
**Objetivo:** Executar a busca bayesiana da MLP densa sobre o espaço de parâmetros justificado teoricamente, utilizando o amostrador TPE e o algoritmo de poda `MedianPruner`.


In [1]:
import sys
import os
for p in ['.', '..', '../..']:
    if os.path.abspath(p) not in sys.path:
        sys.path.insert(0, os.path.abspath(p))

import optuna
import numpy as np
import tensorflow as tf

from src.data_pipeline import (
    load_vra_dataset, clean_and_prepare_vra,
    split_chronological, build_and_fit_preprocessor,
    transform_splits
)
from src.tuning import create_mlp_objective, run_optuna_study


C:\Users\User\Documents\GitHub\deep_learning_trabalho_final\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1. Preparação dos Tensores e Ponderação de Classes


In [2]:
data_path = 'data/raw/vra_amostra_2023.csv' if os.path.exists('data/raw/vra_amostra_2023.csv') else '../data/raw/vra_amostra_2023.csv'
df_raw = load_vra_dataset(data_path)
df_clean = clean_and_prepare_vra(df_raw)
train_df, val_df, test_df = split_chronological(df_clean)

num_cols = ['TempoVooPrevistoMin', 'hora_sin', 'hora_cos', 'dia_semana_sin', 'dia_semana_cos', 'mes_sin', 'mes_cos']
cat_cols = ['ICAOEmpresaAerea', 'CodigoTipoLinha', 'ICAOAerodromoOrigem', 'ICAOAerodromoDestino']
feature_cols = num_cols + cat_cols

preprocessor = build_and_fit_preprocessor(train_df, num_cols, cat_cols)
X_train, y_train, X_val, y_val, X_test, y_test = transform_splits(
    preprocessor, train_df, val_df, test_df, feature_cols, 'target_bin'
)

n_neg, n_pos = int(np.sum(y_train == 0)), int(np.sum(y_train == 1))
class_weights = {0: len(y_train) / (2.0 * n_neg), 1: len(y_train) / (2.0 * n_pos)}
print("Pesos calculados:", class_weights)


Pesos calculados: {0: 0.6369819677317304, 1: 2.3250577367205545}


### 2. Execução do Estudo Optuna com MedianPruner


In [3]:
db_path = 'sqlite:///results/optuna_study.db' if os.path.exists('results') else 'sqlite:///../results/optuna_study.db'
objective = create_mlp_objective(X_train, y_train, X_val, y_val, class_weights=class_weights, max_epochs=10)
study = run_optuna_study(objective, n_trials=5, study_name="notebook_study", storage_path=db_path)

print(f"Melhor Valor de Perda na Validação: {study.best_value:.4f}")
print("Melhores Hiperparâmetros Encontrados:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")


[I 2026-10-10 14:05:13,292] A new study created in RDB with name: notebook_study


[I 2026-10-10 14:05:17,443] Trial 0 finished with value: 0.8301536440849304 and parameters: {'learning_rate': 0.0013292918943162175, 'batch_size': 32, 'n_layers': 1, 'dropout_rate': 0.02904180608409973, 'l2_reg': 0.0029154431891537554, 'units_layer_1': 256}. Best is trial 0 with value: 0.8301536440849304.


[I 2026-10-10 14:05:20,586] Trial 1 finished with value: 0.552141547203064 and parameters: {'learning_rate': 0.03142880890840111, 'batch_size': 256, 'n_layers': 3, 'dropout_rate': 0.21597250932105788, 'l2_reg': 1.461896279370496e-05, 'units_layer_1': 32, 'units_layer_2': 64, 'units_layer_3': 128}. Best is trial 1 with value: 0.552141547203064.


[I 2026-10-10 14:05:23,549] Trial 2 finished with value: 0.7049720883369446 and parameters: {'learning_rate': 0.00015673095467235422, 'batch_size': 64, 'n_layers': 1, 'dropout_rate': 0.34211651325607845, 'l2_reg': 5.762487216478604e-05, 'units_layer_1': 256}. Best is trial 1 with value: 0.552141547203064.


[I 2026-10-10 14:05:27,252] Trial 3 finished with value: 0.7415791749954224 and parameters: {'learning_rate': 0.0005975027999960298, 'batch_size': 32, 'n_layers': 1, 'dropout_rate': 0.4847923138822793, 'l2_reg': 0.001260466458564947, 'units_layer_1': 32}. Best is trial 1 with value: 0.552141547203064.


[I 2026-10-10 14:05:29,759] Trial 4 finished with value: 0.563509464263916 and parameters: {'learning_rate': 0.00018427970406864567, 'batch_size': 256, 'n_layers': 2, 'dropout_rate': 0.41436875457596467, 'l2_reg': 2.67308831078167e-05, 'units_layer_1': 256, 'units_layer_2': 64}. Best is trial 1 with value: 0.552141547203064.


Melhor Valor de Perda na Validação: 0.5521
Melhores Hiperparâmetros Encontrados:
  learning_rate: 0.03142880890840111
  batch_size: 256
  n_layers: 3
  dropout_rate: 0.21597250932105788
  l2_reg: 1.461896279370496e-05
  units_layer_1: 32
  units_layer_2: 64
  units_layer_3: 128
